## **Evaluación de los modelos intra-dominio en Validación (*dev*) y Selección de los Baselines Intra-dominio**

De los modelos intra-dominio entrenados y ajustados en el notebook anterior por cada dataset (MELD e IEMOCAP) con las distintas técnicas de fusión, a continuación los enfrentamos a la partición de validación de su correspondiente dataset para seleccionar aquellos modelos (y por tanto, técnicas de fusión) que alcanzan el mayor rendimiento en *dev*. Estos modelos seleccionados se enfrentarán en notebook posteriores a su correspondiente partición de *test*, se incluirán en el análisis comparativo con el modelo final sobre el corpus global, y se les realizará un análisis de interpretabilidad, estudio de ablación y evaluación de transferencia de aprendizaje. 

Se obtienen las métricas clave de nuestro proyecto (F1-Score Macro, Balanced Accuracy, Recall para estrés, ROC-AUC), entre otras. La selección se lleva a cabo desde el F1-Score Macro, y, ante valores muy ajustados o empate, se tendrá en cuenta el recall. 

In [1]:
# Importamos las librerías necesarias:

import os
import pandas as pd
import re
import subprocess
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
## **Arquitecturas entrenadas con Dataset MELD**

* Modelo de Fusión Temprana:

In [4]:
path_pesos = "pesos/intra_ajuste/early/MELD/pesos_modelo_estres_MELD_early_resnet16_wav2vec11s_roberta32_p256_h64_lr0.0001_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset MELD "
           f"--split dev "
           f"--fusion early "
           f"--video resnet "
           f"--video_frames 16 "
           f"--audio wav2vec "
           f"--audio_len 11 "
           f"--text roberta32 "
           f"--proj_dim 256 "
           f"--hidden_mlp 64 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1108/1108 [00:06<00:00, 168.77it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 6,908,673
Parámetros Entrenables: 6,908,673
Tiempo medio de Inferencia: 4.62 ms / muestra
ROC-AUC Score: 0.7590
F1 Macro: 0.6739
F1 Weighted: 0.8040
Accuracy (en %): 79.60%
Balanced Accuracy (en %): 69.05%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/early/MELD/pesos_modelo_estres_MELD_early_resnet16_wav2vec11s_roberta32_p256_h64_lr0.0001_do0.3.pth --eval_dataset MELD --split dev --fusion early --video resnet --video_frames 16 --audio wav2vec --audio_len 11 --text roberta32 --proj_dim 256 --hidden_mlp 64 --dropout 0.3 ', returncode=0)

* Modelo de Fusión Tardía mediante votación:

In [6]:
path_pesos = "pesos/intra_ajuste/late/voto/MELD/pesos_modelo_estres_MELD_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr1e-05_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset MELD "
           f"--split dev "
           f"--fusion late "
           f"--late_mode promedio "
           f"--video resnet "
           f"--video_frames 16 "
           f"--audio wav2vec "
           f"--audio_len 11 "
           f"--text roberta64 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1108/1108 [00:06<00:00, 164.65it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,994,819
Parámetros Entrenables: 8,994,819
Tiempo medio de Inferencia: 4.75 ms / muestra
ROC-AUC Score: 0.7554
F1 Macro: 0.6051
F1 Weighted: 0.7182
Accuracy (en %): 68.14%
Balanced Accuracy (en %): 68.65%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/voto/MELD/pesos_modelo_estres_MELD_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr1e-05_do0.3.pth --eval_dataset MELD --split dev --fusion late --late_mode promedio --video resnet --video_frames 16 --audio wav2vec --audio_len 11 --text roberta64 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 ', returncode=0)

* Modelo de Fusión Tardía mediante promedio:

In [7]:
path_pesos = "pesos/intra_ajuste/late/promedio/MELD/pesos_modelo_estres_MELD_late_promedio_resnet16_wav2vec7s_roberta64_p256_h64_lr0.0001_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset MELD "
           f"--split dev "
           f"--fusion late "
           f"--late_mode promedio "
           f"--video resnet "
           f"--video_frames 16 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta64 "
           f"--proj_dim 256 "
           f"--hidden_mlp 64 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1108/1108 [00:05<00:00, 217.64it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 6,744,579
Parámetros Entrenables: 6,744,579
Tiempo medio de Inferencia: 3.37 ms / muestra
ROC-AUC Score: 0.7654
F1 Macro: 0.6636
F1 Weighted: 0.7973
Accuracy (en %): 78.88%
Balanced Accuracy (en %): 68.00%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/promedio/MELD/pesos_modelo_estres_MELD_late_promedio_resnet16_wav2vec7s_roberta64_p256_h64_lr0.0001_do0.3.pth --eval_dataset MELD --split dev --fusion late --late_mode promedio --video resnet --video_frames 16 --audio wav2vec --audio_len 7 --text roberta64 --proj_dim 256 --hidden_mlp 64 --dropout 0.3 ', returncode=0)

* Modelo de Fusión Tardía mediante Regresión Logística:

In [8]:
path_pesos = "pesos/intra_ajuste/late/logistica/MELD/pesos_modelo_estres_MELD_late_logistica_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.5.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset MELD "
           f"--split dev "
           f"--fusion late "
           f"--late_mode logistica "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta32 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.5 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1108/1108 [00:05<00:00, 213.00it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,994,823
Parámetros Entrenables: 8,994,823
Tiempo medio de Inferencia: 3.50 ms / muestra
ROC-AUC Score: 0.7854
F1 Macro: 0.6828
F1 Weighted: 0.8033
Accuracy (en %): 79.06%
Balanced Accuracy (en %): 71.38%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/logistica/MELD/pesos_modelo_estres_MELD_late_logistica_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.5.pth --eval_dataset MELD --split dev --fusion late --late_mode logistica --video resnet --video_frames 32 --audio wav2vec --audio_len 7 --text roberta32 --proj_dim 512 --hidden_mlp 128 --dropout 0.5 ', returncode=0)

* Modelo de Fusión mediante Atención:

In [9]:
path_pesos = 'pesos/intra_ajuste/attention/MELD/pesos_modelo_estres_MELD_attention_resnet16_wav2vec7s_roberta64_p512_h128_lr5e-05_do0.3.pth'

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset MELD "
           f"--split dev "
           f"--fusion attention "
           f"--video resnet "
           f"--video_frames 16 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta64 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1108/1108 [00:05<00:00, 214.93it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,928,514
Parámetros Entrenables: 8,928,514
Tiempo medio de Inferencia: 3.43 ms / muestra
ROC-AUC Score: 0.7776
F1 Macro: 0.6791
F1 Weighted: 0.7919
Accuracy (en %): 77.26%
Balanced Accuracy (en %): 73.15%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/attention/MELD/pesos_modelo_estres_MELD_attention_resnet16_wav2vec7s_roberta64_p512_h128_lr5e-05_do0.3.pth --eval_dataset MELD --split dev --fusion attention --video resnet --video_frames 16 --audio wav2vec --audio_len 7 --text roberta64 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 ', returncode=0)

Se muestra una tabla final con los resultados de todos los modelos en validación para las cuatro métricas (F1-Score Macro, Balanced Accuracy, ROC-AUC, Recall para clase Estrés), ordenado por F1:

In [ ]:
rutas_resultados = [
    {
        "nombre": "Fusión Temprana",
        "reporte": "resultados/intra_dev/early/MELD/reporte_pesos_modelo_estres_MELD_early_resnet16_wav2vec11s_roberta32_p256_h64_lr0.0001_do0.3_MELD_dev.txt",
        "modelo": "pesos/intra_ajuste/early/MELD/pesos_modelo_estres_MELD_early_resnet16_wav2vec11s_roberta32_p256_h64_lr0.0001_do0.3.pth"
    },
    {
        "nombre": "Fusión Tardía (Voto)",
        "reporte": "resultados/intra_dev/late/voto/MELD/reporte_pesos_modelo_estres_MELD_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr1e-05_do0.3_MELD_dev.txt",
        "modelo": "pesos/intra_ajuste/late/voto/MELD/pesos_modelo_estres_MELD_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr1e-05_do0.3.pth"
    },
    {
        "nombre": "Fusión Tardía (Promedio)",
        "reporte": "resultados/intra_dev/late/promedio/MELD/reporte_pesos_modelo_estres_MELD_late_promedio_resnet16_wav2vec7s_roberta64_p256_h64_lr0.0001_do0.3_MELD_dev.txt",
        "modelo": "pesos/intra_ajuste/late/promedio/MELD/pesos_modelo_estres_MELD_late_promedio_resnet16_wav2vec7s_roberta64_p256_h64_lr0.0001_do0.3.pth"
    },
    {
        "nombre": "Fusión Tardía (Regresión Logística)",
        "reporte": "resultados/intra_dev/late/logistica/MELD/reporte_pesos_modelo_estres_MELD_late_logistica_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.5_MELD_dev.txt",
        "modelo": "pesos/intra_ajuste/late/logistica/MELD/pesos_modelo_estres_MELD_late_logistica_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.5.pth"
    },
    {
        "nombre": "Fusión mediante Atención",
        "reporte": "resultados/intra_dev/attention/MELD/reporte_pesos_modelo_estres_MELD_attention_resnet16_wav2vec7s_roberta64_p512_h128_lr5e-05_do0.3_MELD_dev.txt",
        "modelo": "pesos/intra_ajuste/attention/MELD/pesos_modelo_estres_MELD_attention_resnet16_wav2vec7s_roberta64_p512_h128_lr5e-05_do0.3.pth"
    } 
]

########### ---------- FUNCIÓN DE EXTRACCIÓN DE HIPERPARÁMETROS DEL NOMBRE DEL MODELO (.pth) ------------##############

def extraer_hiperparametros(nombre_archivo):
    """Extrae tamaño de las ventanas y arquitectura desde el nombre del archivo (.pth)"""
    try:
        v_win = re.search(r'resnet(\d+)', nombre_archivo).group(1) # EJ: resnet32 ---> 32
        a_win = re.search(r'wav2vec(\d+)s', nombre_archivo).group(1) #EJ: wav2vec11s --> 11
        t_win = re.search(r'roberta(\d+)', nombre_archivo).group(1) # EJ: roberta64 --> 64
        proj = re.search(r'_p(\d+)', nombre_archivo).group(1) # EJ: .._p512.. --> 512
        hidden = re.search(r'_h(\d+)', nombre_archivo).group(1) # EJ: .._h256-- --> 256
        lr = re.search(r'_lr([\d.e-]+)', nombre_archivo).group(1) # EJ: .._lr0.0001.. --> 0.0001
        dropout = re.search(r'_do([\d.]+)', nombre_archivo).group(1).rstrip('.') # EJ: .._do0.5. --> 0.5 (sin el punto al final)
        return v_win, a_win, t_win, proj, hidden, lr, dropout
    except AttributeError:
        return "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A"
    
########### ---------- FUNCIÓN DE EXTRACCIÓN DE MÉTRICAS A PARTIR DEL REPORTE .txt ------------##############

def extraer_metricas_txt(ruta_txt):
    """
    Abre el reporte .txt y extrae el F1-Score Macro, Balanced Accuracy, el ROC-AUC y el Recall de la clase 'Estrés'
    """
    f1_macro = 0.0
    auc = 0.0
    ba = 0.0
    recall_estres = 0.0
    
    if not os.path.exists(ruta_txt):
        print(f"No se encuentra el archivo {ruta_txt}")
        return f1_macro, auc, recall_estres
        
    with open(ruta_txt, 'r', encoding='utf-8') as f:
        lineas = f.readlines()
        
    for linea in lineas:
        if "Balanced Accuracy" in linea:
            ba = float(linea.split(":")[1].strip())
        if "ROC-AUC:" in linea:
            auc = float(linea.split(":")[1].strip())
        elif "macro avg" in linea:
            # Ejemplo de línea que puede estar en el reporte: "macro avg    0.80      0.79      0.79       200"
            partes = linea.split()
            f1_macro = float(partes[4]) # El F1-Score está en la 5ª columna
        elif linea.strip().startswith("Estrés"):
            # Ejemplo de línea típica: "   Estrés       0.85      0.82      0.83       100"
            # partes[0]='Estrés', partes[1]=Precision, partes[2]=Recall
            partes = linea.split()
            recall_estres = float(partes[2])
            
    return f1_macro, ba, auc, recall_estres


datos_extraidos = []

for item in rutas_resultados:
    ruta_txt = item["reporte"]
    
    if os.path.exists(ruta_txt):
        # Extraemos métricas del .txt
        f1, ba, auc, recall = extraer_metricas_txt(ruta_txt)
        
        # Extraemos hiperparámetros del nombre del .pth
        v_win, a_win, t_win, proj, hidden, lr, dropout = extraer_hiperparametros(item["modelo"])
        
        datos_extraidos.append({
            "Técnica de Fusión": item["nombre"],
            "Vent. Vídeo (frames)": v_win,
            "Vent. Audio (segundos)": a_win,
            "Vent. Texto (tokens)": t_win,
            "Proj_Dim": proj,
            "Hidden_MLP": hidden,
            "LR": lr,
            "Dropout": dropout,
            "F1-Score Macro": f1,
            "Balanced Accuracy": ba,
            "ROC-AUC": auc,
            "Recall (Estrés)": recall
        })
    else:
        print(f"No se encontró el archivo reporte para {item['nombre']} en: {ruta_txt}")

# Creamos el dataframe y mostramos los resultados:
df_resultados = pd.DataFrame(datos_extraidos)

if not df_resultados.empty:
    # Ordenamos por F1-Score Macro descendente:
    df_resultados = df_resultados.sort_values(by="F1-Score Macro", ascending=False).reset_index(drop=True)
    
    # Exportamos a CSV 
    df_resultados.to_csv("comparativa_final_hiperparametros_MELD_dev.csv", index=False)
    
    # Formato visual para mostrarlo en el notebook:
    columnas_colores = ['F1-Score Macro', 'Balanced Accuracy', 'ROC-AUC', 'Recall (Estrés)']
    styled_df = df_resultados.style.background_gradient(subset=columnas_colores, cmap='Blues')\
                                   .format({
                                       'F1-Score Macro': '{:.4f}',
                                       'Balanced Accuracy': '{:.4f}',
                                       'ROC-AUC': '{:.4f}',
                                       'Recall (Estrés)': '{:.4f}'
                                   })
    display(styled_df)
else:
    print("\nNo se ha generado la tabla.")

,Técnica de Fusión,Vent. Vídeo (frames),Vent. Audio (segundos),Vent. Texto (tokens),Proj_Dim,Hidden_MLP,LR,Dropout,F1-Score Macro,Balanced Accuracy,ROC-AUC,Recall (Estrés)
0,Fusión Tardía (Regresión Logística),32,7,32,512,128,0.0001,0.5,0.6828,0.7138,0.7854,0.5959
1,Fusión mediante Atención,16,7,64,512,128,5e-05,0.3,0.6791,0.7315,0.7776,0.6684
2,Fusión Temprana,16,11,32,256,64,0.0001,0.3,0.6739,0.6905,0.7590,0.5285
3,Fusión Tardía (Promedio),16,7,64,256,64,0.0001,0.3,0.6636,0.6800,0.7654,0.5130
4,Fusión Tardía (Voto),16,11,64,512,128,1e-05,0.3,0.6051,0.6865,0.7554,0.6943


**Conclusión**:

Recordamos el contexto del dataset individual de MELD: se trata de un corpus desbalanceado, donde destacan frases cortas, risas enlatadas de fondo, cortes de cámara constantes, típico de las *sitcoms*. Antes este escenario, las arquitecturas se han dado cuenta de todo esto, y observamos que han adaptado sus hiperparámetros en consecuencia:

- **Técnica de Fusión**: En este escenario, la **Fusión Tardía mediante Regresión Logística** domina en la tabla de resultados, quedando la fusión temprana en el tercer puesto. Esto demuestra que, ante un dataset más ruidoso, procesar las señales de forma independiente antes de ser unificadas permite aislar y filtrar mucho mejor el ruido.
  
- **Ventana de Vídeo (núm. de frames)**: Se muestra una tendencia en reducir el número de frames de 32 a 16, algo que no ocurre en el caso del corpus global (se verá en el próximo notebook). En MELD, como ya hemos mencionado, en los turnos de palabra individuales puede haber cambios de plano abruptos para enfocar las reacciones de terceros o mostrar actores distintos, y la red penaliza estas ventanas más largas que no enriquecen sino que introducen ruido.

- **Ventana de Audio (segundos)**: De forma análoga al vídeo, los dos primeros modelos convergen a ventanas temporales de **7 segundos**. En el EDA realizado para MELD, vimos que la duración media de las frases es muy corta (de aproximadamente 3 segundos). Claramente, 11 segundos implicaría rellenar el tensor con silencio.

- **Ventana de Texto (núm. de tokens)**: Oscila entre 64 y 32. El diálogo en Friends es rápido, con mucha presencia de sarcasmo e ironía. Y para poder capturar esto correctamente, los modelos (en su mayoría) requieren el contexto semántico completo (64 tokens).

- **Arquitectura (`proj_dim` y `hidden_mlp`)**: Observamos que los dos mejores modelos cuentan con una parametrización y espacio latente mayor, para tratar de aprender relaciones mucho más complejas de los datos. 

- **Learning Rate**: El LR en la mayoría converge a 0.0001. 

- **Dropout**: Se consolida en 0.3 para la mayoría, al igual que en el corpus global.

En cuanto a los resultados obtenidos en las cuatro métricas ante la partición de validación (*dev*):

- Los resultados en general disminuyen con respecto al modelo entrenado sobre el dataset global unificado, debido a la naturaleza de MELD, lo que reduce el límite superior de rendimiento de las redes. La **Fusión Tardía (Regresión Logística)** y la **Fusión mediante Atención** están muy próximas en F1-Score Macro y ROC-AUC. Nos llama la atención el alto valor obtenido en recall para la fusión tardía mediante voto.

#### **Decisión Final Modelo Ganador**

Considerando el rendimiento general en F1-Score Macro y su superioridad en Recall, el modelo ganador seleccionado para el corpus de MELD es la **Fusión Tardía mediante Regresión Logística**. Procesar las señales de forma independiente hasta el final para posteriormente ponderar cada una de las predicciones, ha demostrado ser el mecanismo más robusto y seguro.


---
## **Arquitecturas entrenadas con Dataset IEMOCAP**

* Modelo de Fusión Temprana:

In [11]:
path_pesos = "pesos/intra_ajuste/early/IEMOCAP/pesos_modelo_estres_IEMOCAP_early_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset IEMOCAP "
           f"--split dev "
           f"--fusion early "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta32 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1370/1370 [00:06<00:00, 218.89it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 9,650,689
Parámetros Entrenables: 9,650,689
Tiempo medio de Inferencia: 3.38 ms / muestra
ROC-AUC Score: 0.7967
F1 Macro: 0.7294
F1 Weighted: 0.7589
Accuracy (en %): 76.20%
Balanced Accuracy (en %): 72.43%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/early/IEMOCAP/pesos_modelo_estres_IEMOCAP_early_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.3.pth --eval_dataset IEMOCAP --split dev --fusion early --video resnet --video_frames 32 --audio wav2vec --audio_len 7 --text roberta32 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 ', returncode=0)

* Modelo de Fusión Tardía mediante votación:

In [12]:
path_pesos = "pesos/intra_ajuste/late/voto/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr5e-05_do0.5.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset IEMOCAP "
           f"--split dev "
           f"--fusion late "
           f"--late_mode promedio "
           f"--video resnet "
           f"--video_frames 16 "
           f"--audio wav2vec "
           f"--audio_len 11 "
           f"--text roberta64 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.5 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1370/1370 [00:08<00:00, 162.42it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,994,819
Parámetros Entrenables: 8,994,819
Tiempo medio de Inferencia: 4.80 ms / muestra
ROC-AUC Score: 0.7727
F1 Macro: 0.7037
F1 Weighted: 0.7370
Accuracy (en %): 74.16%
Balanced Accuracy (en %): 69.81%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/voto/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr5e-05_do0.5.pth --eval_dataset IEMOCAP --split dev --fusion late --late_mode promedio --video resnet --video_frames 16 --audio wav2vec --audio_len 11 --text roberta64 --proj_dim 512 --hidden_mlp 128 --dropout 0.5 ', returncode=0)

* Modelo de Fusión Tardía mediante promedio:

In [13]:
path_pesos = "pesos/intra_ajuste/late/promedio/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_promedio_resnet32_wav2vec11s_roberta64_p512_h128_lr0.0001_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset IEMOCAP "
           f"--split dev "
           f"--fusion late "
           f"--late_mode promedio "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 11 "
           f"--text roberta64 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1370/1370 [00:08<00:00, 163.61it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,994,819
Parámetros Entrenables: 8,994,819
Tiempo medio de Inferencia: 4.80 ms / muestra
ROC-AUC Score: 0.7927
F1 Macro: 0.7183
F1 Weighted: 0.7464
Accuracy (en %): 74.67%
Balanced Accuracy (en %): 71.78%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/promedio/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_promedio_resnet32_wav2vec11s_roberta64_p512_h128_lr0.0001_do0.3.pth --eval_dataset IEMOCAP --split dev --fusion late --late_mode promedio --video resnet --video_frames 32 --audio wav2vec --audio_len 11 --text roberta64 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 ', returncode=0)

* Modelo de Fusión Tardía mediante Regresión Logística:

In [14]:
path_pesos = "pesos/intra_ajuste/late/logistica/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_logistica_resnet32_wav2vec7s_roberta64_p256_h64_lr1e-05_do0.3.pth"

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset IEMOCAP "
           f"--split dev "
           f"--fusion late "
           f"--late_mode logistica "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 7 "
           f"--text roberta64 "
           f"--proj_dim 256 "
           f"--hidden_mlp 64 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1370/1370 [00:07<00:00, 172.61it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 6,744,583
Parámetros Entrenables: 6,744,583
Tiempo medio de Inferencia: 4.24 ms / muestra
ROC-AUC Score: 0.8049
F1 Macro: 0.7175
F1 Weighted: 0.7350
Accuracy (en %): 72.85%
Balanced Accuracy (en %): 73.95%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/late/logistica/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_logistica_resnet32_wav2vec7s_roberta64_p256_h64_lr1e-05_do0.3.pth --eval_dataset IEMOCAP --split dev --fusion late --late_mode logistica --video resnet --video_frames 32 --audio wav2vec --audio_len 7 --text roberta64 --proj_dim 256 --hidden_mlp 64 --dropout 0.3 ', returncode=0)

* Modelo de Fusión mediante Atención:

In [15]:
path_pesos = 'pesos/intra_ajuste/attention/IEMOCAP/pesos_modelo_estres_IEMOCAP_attention_resnet32_wav2vec11s_roberta32_p512_h128_lr5e-05_do0.3.pth'

comando = (f"python evaluate.py "
           f"--model_path {path_pesos} "
           f"--eval_dataset IEMOCAP "
           f"--split dev "
           f"--fusion attention "
           f"--video resnet "
           f"--video_frames 32 "
           f"--audio wav2vec "
           f"--audio_len 11 "
           f"--text roberta32 "
           f"--proj_dim 512 "
           f"--hidden_mlp 128 "
           f"--dropout 0.3 ")
                        
subprocess.run(comando, shell=True)

100%|██████████| 1370/1370 [00:09<00:00, 150.51it/s]


MÉTRICAS DEL MODELO EN DEV:
Parámetros Totales: 8,928,514
Parámetros Entrenables: 8,928,514
Tiempo medio de Inferencia: 5.20 ms / muestra
ROC-AUC Score: 0.7841
F1 Macro: 0.7273
F1 Weighted: 0.7533
Accuracy (en %): 75.26%
Balanced Accuracy (en %): 72.88%


CompletedProcess(args='python evaluate.py --model_path pesos/intra_ajuste/attention/IEMOCAP/pesos_modelo_estres_IEMOCAP_attention_resnet32_wav2vec11s_roberta32_p512_h128_lr5e-05_do0.3.pth --eval_dataset IEMOCAP --split dev --fusion attention --video resnet --video_frames 32 --audio wav2vec --audio_len 11 --text roberta32 --proj_dim 512 --hidden_mlp 128 --dropout 0.3 ', returncode=0)

Se muestra una tabla final con los resultados de todos los modelos en validación para las cuatro métricas (F1-Score Macro, Balanced Accuracy, ROC-AUC, Recall para clase Estrés), ordenado por F1:

In [3]:
rutas_resultados = [
    {
        "nombre": "Fusión Temprana",
        "reporte": "resultados/intra_dev/early/IEMOCAP/reporte_pesos_modelo_estres_IEMOCAP_early_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.3_IEMOCAP_dev.txt",
        "modelo": "pesos/intra_ajuste/early/IEMOCAP/pesos_modelo_estres_IEMOCAP_early_resnet32_wav2vec7s_roberta32_p512_h128_lr0.0001_do0.3.pth"
    },
    {
        "nombre": "Fusión Tardía (Voto)",
        "reporte": "resultados/intra_dev/late/voto/IEMOCAP/reporte_pesos_modelo_estres_IEMOCAP_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr5e-05_do0.5_IEMOCAP_dev.txt",
        "modelo": "pesos/intra_ajuste/late/voto/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_voto_resnet16_wav2vec11s_roberta64_p512_h128_lr5e-05_do0.5.pth"
    },
    {
        "nombre": "Fusión Tardía (Promedio)",
        "reporte": "resultados/intra_dev/late/promedio/IEMOCAP/reporte_pesos_modelo_estres_IEMOCAP_late_promedio_resnet32_wav2vec11s_roberta64_p512_h128_lr0.0001_do0.3_IEMOCAP_dev.txt",
        "modelo": "pesos/intra_ajuste/late/promedio/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_promedio_resnet32_wav2vec11s_roberta64_p512_h128_lr0.0001_do0.3.pth"
    },
    {
        "nombre": "Fusión Tardía (Regresión Logística)",
        "reporte": "resultados/intra_dev/late/logistica/IEMOCAP/reporte_pesos_modelo_estres_IEMOCAP_late_logistica_resnet32_wav2vec7s_roberta64_p256_h64_lr1e-05_do0.3_IEMOCAP_dev.txt",
        "modelo": "pesos/intra_ajuste/late/logistica/IEMOCAP/pesos_modelo_estres_IEMOCAP_late_logistica_resnet32_wav2vec7s_roberta64_p256_h64_lr1e-05_do0.3.pth"
    },
    {
        "nombre": "Fusión mediante Atención",
        "reporte": "resultados/intra_dev/attention/IEMOCAP/reporte_pesos_modelo_estres_IEMOCAP_attention_resnet32_wav2vec11s_roberta32_p512_h128_lr5e-05_do0.3_IEMOCAP_dev.txt",
        "modelo": "pesos/intra_ajuste/attention/IEMOCAP/pesos_modelo_estres_IEMOCAP_attention_resnet32_wav2vec11s_roberta32_p512_h128_lr5e-05_do0.3.pth"
    } 
]


datos_extraidos = []

for item in rutas_resultados:
    ruta_txt = item["reporte"]
    
    if os.path.exists(ruta_txt):
        # Extraemos métricas del .txt
        f1, ba, auc, recall = extraer_metricas_txt(ruta_txt)
        
        # Extraemos hiperparámetros del nombre del .pth
        v_win, a_win, t_win, proj, hidden, lr, dropout = extraer_hiperparametros(item["modelo"])
        
        datos_extraidos.append({
            "Técnica de Fusión": item["nombre"],
            "Vent. Vídeo (frames)": v_win,
            "Vent. Audio (segundos)": a_win,
            "Vent. Texto (tokens)": t_win,
            "Proj_Dim": proj,
            "Hidden_MLP": hidden,
            "LR": lr,
            "Dropout": dropout,
            "F1-Score Macro": f1,
            "Balanced Accuracy": ba,
            "ROC-AUC": auc,
            "Recall (Estrés)": recall
        })
    else:
        print(f"No se encontró el archivo reporte para {item['nombre']} en: {ruta_txt}")

# Creamos el dataframe y mostramos los resultados:
df_resultados = pd.DataFrame(datos_extraidos)

if not df_resultados.empty:
    # Ordenamos por F1-Score Macro descendente:
    df_resultados = df_resultados.sort_values(by="F1-Score Macro", ascending=False).reset_index(drop=True)
    
    # Exportamos a CSV 
    df_resultados.to_csv("comparativa_final_hiperparametros_IEMOCAP_dev.csv", index=False)
    
    # Formato visual para mostrarlo en el notebook:
    columnas_colores = ['F1-Score Macro', 'Balanced Accuracy', 'ROC-AUC', 'Recall (Estrés)']
    styled_df = df_resultados.style.background_gradient(subset=columnas_colores, cmap='Blues')\
                                   .format({
                                       'F1-Score Macro': '{:.4f}',
                                       'Balanced Accuracy': '{:.4f}',
                                       'ROC-AUC': '{:.4f}',
                                       'Recall (Estrés)': '{:.4f}'
                                   })
    display(styled_df)
else:
    print("\nNo se ha generado la tabla.")

,Técnica de Fusión,Vent. Vídeo (frames),Vent. Audio (segundos),Vent. Texto (tokens),Proj_Dim,Hidden_MLP,LR,Dropout,F1-Score Macro,Balanced Accuracy,ROC-AUC,Recall (Estrés)
0,Fusión Temprana,32,7,32,512,128,0.0001,0.3,0.7294,0.7243,0.7967,0.6043
1,Fusión mediante Atención,32,11,32,512,128,5e-05,0.3,0.7273,0.7288,0.7841,0.6532
2,Fusión Tardía (Promedio),32,11,64,512,128,0.0001,0.3,0.7183,0.7178,0.7927,0.6255
3,Fusión Tardía (Regresión Logística),32,7,64,256,64,1e-05,0.3,0.7175,0.7395,0.8049,0.7745
4,Fusión Tardía (Voto),16,11,64,512,128,5e-05,0.5,0.7037,0.6981,0.7727,0.5596


**Conclusión**:

Recordamos el contexto del dataset de IEMOCAP: es un entorno de laboratorio controlado, con actores profesionales, sin interrupciones abruptas y emociones actuadas de forma sostenida. Es todo lo contrario a MELD y, por tanto, el comportamiento y ajuste de los modelos ha dado un giro de 180 grados con respecto a los modelos en MELD. 

- **Técnica de Fusión**: Ocurre todo lo contrario que en MELD, y es que la **Fusión Temprana** resulta ser la mejor arquitectura, seguida muy cerca por la Fusión mediante Atención. Este fenómeno se debe a que, al estar constituido IEMOCAP por actuaciones profesionales, las tres modalidades se encuentran alineadas. Por ejemplo, si un actor simula estrés o enfado, su ceño (vídeo), su tono de voz (audio) y su vocabulario (texto) se alteran simultáneamente, todos reflejando dicho estado emocional. La Fusión Temprana aprovecha esta correlación, concatenando las características desde el principio para crear una representación consistente, algo que no ocurría en el entorno ruidoso de MELD (en el cual, para enmascarar ese ruido excesivo y modalidades destrozadas por el formato televisivo rápido, se necesitaba una fusión a nivel de decisión).

- **Ventana de Vídeo (núm. de frames)**: De nuevo, vuelve a ocurrir todo lo contrario a MELD. Al no haber cortes abruptos, y recibir información de los movimientos y expresiones de ambos actores simultáneamente, los modelos se ven mucho más beneficiados por contextos mayores (de 32 frames), en lugar de limitarlo a 16 frames, ya que un contexto mayor no supone más ruido sino permite capturar mucho mejor las expresiones faciales y movimiento corporal.

- **Ventana de Audio (segundos)**: Se observa una división entre 7 y 11 segundos, aunque el modelo ganador presenta 7 segundos como ventana de audio. En IEMOCAP, a pesar de presentar turnos de palabra mucho más largos que MELD, 7 segundos sigue siendo el límite óptimo (con 11 segundos se puede introducir silencios prolongados).

- **Ventana de Texto (núm. de tokens)**: Los mejores modelos presentan ventanas de contexto más limitadas en tokens (32), al igual que en MELD. 

- **Arquitectura (`proj_dim` y `hidden_mlp`)**: Los modelos se ven beneficiados por una mayor capacidad paramétrica, dado que IEMOCAP es un dataset mucho más limpio, sin ruido de fondo y entorno controlado, la red permite modelar relaciones multimodales más complejas de la actuación humana. 

- **Learning Rate**: Al ser un dataset tan limpio y correlacionado, permite que el mejor modelo converja a un LR mayor (de 0.0001) sin riesgo de divergencia.

- **Dropout**: Se mantiene en 0.3 en prácticamente todos los modelos. 

En cuanto a los resultados obtenidos en las cuatro métricas ante la partición de validación (*dev*):

- La **Fusión Temprana** domina el F1-Score Macro y alcanza el Recall más alto de todos los modelos para la partición de validación. En general, las métricas son ligeramente mejores a las obtenidas de los modelos entrenados sobre el corpus global unificado, debido a la naturaleza limpia y controlada de IEMOCAP (esto se verá en el posterior análisis comparativo).

**NOTA**: Nos llama la atención el valor atípico alcanzado en Fusión Tardía por Voto, que alcanza en Recall un valor excesivamente alto, lo que nos indica que mediante voto mayoritario el modelo tiende a clasificar todo como estrés para evitar fallar, lo que dispara los falsos positivos. 

#### **Decisión Final Modelo Ganador**

El modelo finalmente seleccionado como ganador para este dataset de IEMOCAP es el obtenido mediante **Fusión Temprana**. Este corpus demuestra que, cuando las señales de audio, vídeo y texto son de alta calidad y cuentan con una sincronía expresiva perfecta, fusionar las representaciones en las primeras etapas de la arquitectura es la estrategia más eficaz.